# 駅のまわりを広げると、似た街は変わる？
500・600・700・800mは駅から東西南北への半幅です。一辺は1.0〜1.6km。最適な商圏を決めず、分析条件への感度を調べます。

[固定仕様](../docs/aoi-sensitivity-design.md) / [方法](../docs/aoi-sensitivity.md) / [検証](../docs/aoi-sensitivity-validation.md)

Aは完全250m区画だけから最大密度・集中度を算出する追加品質確認版です。密度と構成比はAOI全域。旧Aを監査として保持し、B/Cは保存済み結果を使用します。元分析は計算方法の初回手順を参照。500mは旧Aと一致します。600/700mで最大密度を探す範囲は中央1km四方、800mでは中央1.5km四方です。

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')
from geoai_open_lab.aoi_full_cells import run
from geoai_open_lab.aoi_sensitivity_figures import figures
from IPython.display import display, SVG
import pandas as pd

## 1. 同じ入力・同じ候補集合で比較
保存済み監査値からAだけを再計算し、500mの特徴量・全順位・距離を照合します。候補は800mで重なる駅を全条件で除外。Bは固定961カテゴリTF-IDF、CはSVD64＋L2の既存結果を使用します。旧分析では各範囲で学習したため、Cの軸を直接比較しません。表のold_vs_500は旧A、full_vs_500は修正版A、full_vs_old_same_widthは同じ範囲での新旧Aの比較です。

In [ ]:
summary, manifest, out = run()
paths = figures(out)
display(summary.round(3))

## 2. 吉祥寺に似ている駅
ランキングの距離をA/B/C間で直接比較しません。共通数は駅グループIDで数えます。

In [ ]:
rankings = pd.read_csv(out/'rankings.csv')
display(rankings[(rankings['rank'] <= 10) & (rankings.half_width_m == 800)].pivot(index='rank', columns='mode', values='station_name'))
display(SVG(filename=str(out/'target_rankings.svg')))

## 3. 全駅の分布と3群分類
平均だけでなく分布を見ます。ARIは分類の一致度で、正解率ではありません。群番号は最大一致で対応させます。

In [ ]:
display(SVG(filename=str(out/'common_distribution.svg')))
display(SVG(filename=str(out/'cluster_changes.svg')))

## 4. 限界
旧分析ではAOIごとにIDF・SVDを学習しましたが、今回の追加確認ではB/Cを再学習していません。Aの標準化は各AOIで再計算します。完全区画だけにしても500→800mの上位10共通数は平均2.24駅で、範囲による感度は残りました。旧A（平均1.97駅）は監査用に保持します。近接駅は同じPOIを共有し、独立した街ではありません。増えたカテゴリは順位変化の原因と断定しません。

正解となる街の範囲・類似ラベルはなく、範囲を広げるほど精度が上がるとは言えません。円・徒歩圏との比較は未実施です。